# 01b — Synthesize Q1 2026 sales

Raw sales end 2025-12-31, but the inventory snapshot and PO receipts extend through ~2026-04-15. Without this step the inventory rewind sees a 15-week flat plateau. Here we forward-synthesize weekly sales rows for active lanes using historical run-rate, and classify every lane (active / stocked_out / inactive / discontinued / anomaly).

**Upstream:** `sales.parquet`, `po.parquet`, `inv_snapshot.parquet`

**Output:** `sales_synth.parquet`, `lane_status.parquet`

**Consumed by:** `04_inventory_rewind.ipynb` (unions sales ∪ sales_synth before rewind); `09_reorder_alerts.ipynb` (reads lane_status for UI badges).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "pipeline" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ART = ROOT / "pipeline" / "artifacts"

from src.load import load_cached
from src.synthesize_q1 import synthesize_q1_sales, DEFAULT_AS_OF, DEFAULT_GAP_END, DEFAULT_LOOKBACK_WEEKS

## 2. Load upstream

In [ ]:
c = load_cached()
sales = c["sales"]
po = c["po"]
inv_snap = c["inv_snapshot"]

DC_MAP = {"1": "SF", "2": "NJ", "3": "LA"}

print(f"sales rows      : {len(sales):,}  max DOCDATE: {pd.to_datetime(sales.DOCDATE).max().date()}")
print(f"po rows         : {len(po):,}  max Receipt : {pd.to_datetime(po[\'Receipt Date\']).max().date()}")
print(f"inv_snap rows   : {len(inv_snap):,}")
print(f"AS_OF={DEFAULT_AS_OF.date()}  GAP_END={DEFAULT_GAP_END.date()}  lookback={DEFAULT_LOOKBACK_WEEKS}w")

## 3. Do the work

In [ ]:
sales_synth, lane_status = synthesize_q1_sales(
    sales=sales, po=po, inv_snap=inv_snap, dc_map=DC_MAP,
)

print(f"sales_synth rows: {len(sales_synth):,}")
print(f"lane_status rows: {len(lane_status):,}")
print()
print(lane_status["status"].value_counts())

## 4. Validate

In [ ]:
assert (sales_synth["synthetic"] == True).all(), "all synth rows must be flagged"
assert set(sales_synth.columns) >= {"DOCDATE","ITEMNMBR","LOCNCODE","QUANTITY_adj","QTYBSUOM","synthetic"}
assert sales_synth["DOCDATE"].min() > DEFAULT_AS_OF, "synth must be strictly after as_of"
assert sales_synth["DOCDATE"].max() <= DEFAULT_GAP_END, "synth must end by gap_end"

missing = set(sales.columns) - set(sales_synth.columns)
assert not missing, f"synth missing cols present in sales: {missing}"

qty_base = (sales_synth["QUANTITY_adj"] * sales_synth["QTYBSUOM"]).sum()
print(f"synthetic QTY_BASE total: {qty_base:,.0f}")
per_dc = sales_synth.groupby("LOCNCODE").apply(lambda d: (d.QUANTITY_adj * d.QTYBSUOM).sum()).to_dict()
print(f"per DC: {per_dc}")
print(f"week range: {sales_synth.DOCDATE.min().date()} -> {sales_synth.DOCDATE.max().date()}")

## 5. Save

In [ ]:
sales_synth.to_parquet(ART/"sales_synth.parquet")
lane_status.to_parquet(ART/"lane_status.parquet")
print(f"wrote sales_synth.parquet  ({len(sales_synth):,} rows)")
print(f"wrote lane_status.parquet  ({len(lane_status):,} rows)")

## 6. Promote

Logic already lives in `src/synthesize_q1.py`. Downstream notebooks consume via:

```python
sales_synth = pd.read_parquet(ART/"sales_synth.parquet")
lane_status = pd.read_parquet(ART/"lane_status.parquet")
sales_full  = pd.concat([sales, sales_synth], ignore_index=True)
```

`04_inventory_rewind.ipynb` must use `sales_full` in `build_inv_weekly`. `09_reorder_alerts.ipynb` reads `lane_status` for UI badges. Keep `clean_demand_weekly.parquet` built from `sales` only (no synth), so the forecaster does not train on estimates.